In [29]:
!pip install pandas scikit-learn streamlit -q
import re
import io
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from google.colab import files

In [30]:
SKILLS_DATABASE = [
    # Programming Languages
    "python", "java", "c++", "c", "javascript", "typescript",
    "r", "sql", "html", "css",

    # AI / Machine Learning
    "machine learning", "deep learning", "artificial intelligence",
    "natural language processing", "nlp", "computer vision",
    "data science", "generative ai", "llm",
    "large language models", "transformers",

    # ML Libraries
    "scikit-learn", "sklearn", "tensorflow", "pytorch",
    "keras", "xgboost",

    # Data Analysis
    "pandas", "numpy", "matplotlib", "seaborn",
    "excel", "power bi", "tableau",

    # Web / Application Development
    "streamlit", "flask", "django", "fastapi",
    "react", "node.js",

    # Databases
    "mysql", "postgresql", "mongodb", "sqlite",

    # Cloud / DevOps
    "aws", "azure", "google cloud", "gcp",
    "docker", "kubernetes", "git", "github",

    # Data Engineering
    "apache spark", "spark", "hadoop", "airflow", "etl",

    # Other
    "api", "rest api", "linux", "statistics",
    "data visualization"
]

In [31]:
def clean_text(text):
    """Clean and normalize text."""
    text = str(text).lower()
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def extract_skills(text):
    """Extract skills found in text."""

    text = clean_text(text)
    found_skills = []

    for skill in SKILLS_DATABASE:
        pattern = r"(?<!\w)" + re.escape(skill.lower()) + r"(?!\w)"

        if re.search(pattern, text):
            found_skills.append(skill)

    return sorted(set(found_skills))


def extract_name(text, filename=""):
    """Try to extract candidate name from resume text."""

    lines = [
        line.strip()
        for line in str(text).split("\n")
        if line.strip()
    ]

    skip_words = [
        "resume", "curriculum vitae", "email", "phone",
        "skills", "education", "experience",
        "summary", "profile", "linkedin"
    ]

    for line in lines[:8]:

        if not any(word in line.lower() for word in skip_words):

            # Remove special characters
            possible_name = re.sub(
                r"[^a-zA-Z\s]",
                "",
                line
            ).strip()

            words = possible_name.split()

            if 2 <= len(words) <= 4:
                return possible_name.title()

    # Fallback to filename
    if filename:
        name = filename.rsplit(".", 1)[0]
        name = re.sub(r"candidate[_-]?", "", name, flags=re.I)
        name = name.replace("_", " ").replace("-", " ").strip()

        if name:
            return name.title()

    return "Unknown"


def extract_experience(text):
    """Extract years of experience."""

    text = clean_text(text)

    patterns = [
        r"(\d+(?:\.\d+)?)\+?\s*years?\s+of\s+experience",
        r"(\d+(?:\.\d+)?)\+?\s*years?\s+experience",
        r"experience\s*[:\-]?\s*(\d+(?:\.\d+)?)\+?\s*years?"
    ]

    years_found = []

    for pattern in patterns:

        matches = re.findall(pattern, text)

        for match in matches:
            try:
                years_found.append(float(match))
            except ValueError:
                pass

    return max(years_found) if years_found else 0.0


def extract_education(text):
    """Extract highest education level."""

    text = clean_text(text)

    education_levels = [
        (
            ["phd", "ph.d", "doctorate"],
            "PhD"
        ),
        (
            ["master", "master's", "m.tech", "mtech", "m.sc",
             "msc", "mba"],
            "Master's"
        ),
        (
            ["bachelor", "bachelor's", "b.tech", "btech",
             "b.e", "b.sc", "bsc", "bca"],
            "Bachelor's"
        ),
        (
            ["diploma"],
            "Diploma"
        )
    ]

    for keywords, level in education_levels:
        if any(keyword in text for keyword in keywords):
            return level

    return "Not Found"


def parse_resume(text, filename=""):
    """Extract all candidate information."""

    return {
        "name": extract_name(text, filename),
        "skills": extract_skills(text),
        "experience": extract_experience(text),
        "education": extract_education(text),
        "raw_text": str(text)
    }

In [32]:
def calculate_skill_match(resume_skills, job_skills):

    if not job_skills:
        return 0.0, [], []

    resume_set = {skill.lower() for skill in resume_skills}
    job_set = {skill.lower() for skill in job_skills}

    matched_skills = resume_set.intersection(job_set)
    missing_skills = job_set.difference(resume_set)

    score = (
        len(matched_skills) / len(job_set)
    ) * 100

    return (
        round(score, 2),
        sorted(matched_skills),
        sorted(missing_skills)
    )


def extract_required_experience(job_description):
    """Extract required years of experience from JD."""

    text = clean_text(job_description)

    patterns = [
        r"(\d+(?:\.\d+)?)\+?\s*years?\s+of\s+experience",
        r"(\d+(?:\.\d+)?)\+?\s*years?\s+experience",
        r"minimum\s+(\d+(?:\.\d+)?)\+?\s*years?",
        r"at least\s+(\d+(?:\.\d+)?)\+?\s*years?"
    ]

    years_found = []

    for pattern in patterns:

        matches = re.findall(pattern, text)

        for match in matches:
            try:
                years_found.append(float(match))
            except ValueError:
                pass

    return max(years_found) if years_found else 0.0


def calculate_experience_match(
    candidate_experience,
    required_experience
):
    """Calculate experience compatibility."""

    if required_experience == 0:
        return 100.0

    score = min(
        candidate_experience / required_experience,
        1
    ) * 100

    return round(score, 2)


def calculate_education_match(
    candidate_education,
    job_description
):
    """Calculate education compatibility."""

    text = clean_text(job_description)

    education_rank = {
        "Not Found": 0,
        "Diploma": 1,
        "Bachelor's": 2,
        "Master's": 3,
        "PhD": 4
    }

    required_rank = 0

    if any(
        keyword in text
        for keyword in ["phd", "ph.d", "doctorate"]
    ):
        required_rank = 4

    elif any(
        keyword in text
        for keyword in [
            "master", "master's", "m.tech",
            "mtech", "m.sc", "msc", "mba"
        ]
    ):
        required_rank = 3

    elif any(
        keyword in text
        for keyword in [
            "bachelor", "bachelor's", "b.tech",
            "btech", "b.e", "b.sc", "bsc", "bca"
        ]
    ):
        required_rank = 2

    # No specific education requirement
    if required_rank == 0:
        return 100.0

    candidate_rank = education_rank.get(
        candidate_education,
        0
    )

    if candidate_rank >= required_rank:
        return 100.0

    return 50.0


def calculate_tfidf_similarity(
    resume_text,
    job_description
):
    """Calculate AI-based semantic text similarity."""

    if (
        not str(resume_text).strip()
        or not str(job_description).strip()
    ):
        return 0.0

    try:

        vectorizer = TfidfVectorizer(
            stop_words="english"
        )

        tfidf_matrix = vectorizer.fit_transform([
            str(resume_text),
            str(job_description)
        ])

        similarity = cosine_similarity(
            tfidf_matrix[0:1],
            tfidf_matrix[1:2]
        )[0][0]

        return round(similarity * 100, 2)

    except ValueError:
        return 0.0


def calculate_match_score(
    resume_data,
    job_description
):
    """Calculate the final weighted Resume Match Score."""

    job_skills = extract_skills(job_description)

    required_experience = extract_required_experience(
        job_description
    )

    # Individual scores
    skill_score, matched_skills, missing_skills = (
        calculate_skill_match(
            resume_data["skills"],
            job_skills
        )
    )

    experience_score = calculate_experience_match(
        resume_data["experience"],
        required_experience
    )

    education_score = calculate_education_match(
        resume_data["education"],
        job_description
    )

    similarity_score = calculate_tfidf_similarity(
        resume_data["raw_text"],
        job_description
    )

    # Weighted final score
    final_score = (
        skill_score * 0.50
        + experience_score * 0.25
        + education_score * 0.15
        + similarity_score * 0.10
    )

    return {
        "match_score": round(final_score, 2),
        "skill_score": skill_score,
        "experience_score": experience_score,
        "education_score": education_score,
        "similarity_score": similarity_score,
        "matched_skills": matched_skills,
        "missing_skills": missing_skills,
        "job_skills": job_skills,
        "required_experience": required_experience
    }

In [33]:
def process_txt_file(filename, file_content):
    """
    Process one TXT file as one candidate.
    """

    try:
        text = file_content.decode(
            "utf-8",
            errors="ignore"
        )

        return [
            parse_resume(
                text,
                filename
            )
        ]

    except Exception as e:
        print(f"Error reading {filename}: {e}")
        return []


def find_column(columns, possible_names):
    """
    Find a matching column name in a CSV.
    """

    normalized_columns = {
        col.lower().strip(): col
        for col in columns
    }

    for name in possible_names:

        if name in normalized_columns:
            return normalized_columns[name]

    return None


def process_csv_file(filename, file_content):
    """
    Process each CSV row as one candidate.
    """

    try:

        df = pd.read_csv(
            io.BytesIO(file_content)
        )

        # Remove completely empty rows
        df = df.dropna(how="all")

        candidates = []

        # Try to identify useful columns
        name_col = find_column(
            df.columns,
            ["name", "candidate name", "candidate"]
        )

        skills_col = find_column(
            df.columns,
            ["skills", "technical skills", "skill"]
        )

        experience_col = find_column(
            df.columns,
            [
                "experience",
                "years of experience",
                "experience years",
                "years"
            ]
        )

        education_col = find_column(
            df.columns,
            ["education", "qualification", "degree"]
        )

        for index, row in df.iterrows():

            # Convert entire row into text
            row_text = " ".join(
                str(value)
                for value in row.values
                if pd.notna(value)
            )

            # Parse normally first
            candidate = parse_resume(
                row_text,
                f"{filename}_row_{index + 1}"
            )

            # Use structured CSV values when available
            if name_col and pd.notna(row[name_col]):
                candidate["name"] = str(
                    row[name_col]
                ).strip()

            if skills_col and pd.notna(row[skills_col]):

                skills_text = str(row[skills_col])

                candidate["skills"] = extract_skills(
                    skills_text
                )

            if (
                experience_col
                and pd.notna(row[experience_col])
            ):

                exp_text = str(
                    row[experience_col]
                )

                exp_match = re.search(
                    r"(\d+(?:\.\d+)?)",
                    exp_text
                )

                if exp_match:
                    candidate["experience"] = float(
                        exp_match.group(1)
                    )

            if (
                education_col
                and pd.notna(row[education_col])
            ):

                candidate["education"] = (
                    extract_education(
                        str(row[education_col])
                    )
                )

            candidates.append(candidate)

        return candidates

    except Exception as e:

        print(
            f"Error processing CSV {filename}: {e}"
        )

        return []


def process_uploaded_files(uploaded_files):
    """
    Process multiple TXT and CSV files.
    """

    all_candidates = []

    for filename, file_content in uploaded_files.items():

        if filename.lower().endswith(".txt"):

            candidates = process_txt_file(
                filename,
                file_content
            )

            all_candidates.extend(candidates)

        elif filename.lower().endswith(".csv"):

            candidates = process_csv_file(
                filename,
                file_content
            )

            all_candidates.extend(candidates)

        else:

            print(
                f"Skipping unsupported file: {filename}"
            )

    return all_candidates

In [44]:
print(
    "Upload one or more resume files "
    "(.txt or .csv)"
)

uploaded_files = files.upload()

print(
    f"\n{len(uploaded_files)} file(s) uploaded."
)

for filename in uploaded_files:
    print("✓", filename)

Upload one or more resume files (.txt or .csv)


Saving candidate_1.txt to candidate_1 (4).txt
Saving candidate_2.txt to candidate_2 (3).txt
Saving candidate_3.txt to candidate_3 (3).txt
Saving candidate_4.txt to candidate_4 (2).txt
Saving candidate_5.txt to candidate_5 (2).txt
Saving candidates.csv to candidates (3).csv

6 file(s) uploaded.
✓ candidate_1 (4).txt
✓ candidate_2 (3).txt
✓ candidate_3 (3).txt
✓ candidate_4 (2).txt
✓ candidate_5 (2).txt
✓ candidates (3).csv


In [45]:
job_description = """
We are looking for a Machine Learning Engineer
with at least 2 years of experience.

Required Skills:
Python, SQL, Machine Learning, Deep Learning,
Scikit-learn, TensorFlow, Pandas, AWS and Docker.

The candidate should have experience in developing,
training and deploying machine learning models.

Education:
Bachelor's or Master's degree in Computer Science,
Artificial Intelligence, Data Science or a related field.
"""

print("JOB DESCRIPTION:")
print(job_description)

JOB DESCRIPTION:

We are looking for a Machine Learning Engineer
with at least 2 years of experience.

Required Skills:
Python, SQL, Machine Learning, Deep Learning,
Scikit-learn, TensorFlow, Pandas, AWS and Docker.

The candidate should have experience in developing,
training and deploying machine learning models.

Education:
Bachelor's or Master's degree in Computer Science,
Artificial Intelligence, Data Science or a related field.



In [46]:
# Process all uploaded files
candidates = process_uploaded_files(
    uploaded_files
)

print(
    f"\nTotal candidates found: "
    f"{len(candidates)}"
)


results = []

for candidate in candidates:

    # Calculate resume match
    match_data = calculate_match_score(
        candidate,
        job_description
    )

    result = {
        "Name": candidate["name"],
        "Experience (Years)": candidate["experience"],
        "Education": candidate["education"],
        "Skills": ", ".join(candidate["skills"]),
        "Matched Skills": ", ".join(
            match_data["matched_skills"]
        ),
        "Missing Skills": ", ".join(
            match_data["missing_skills"]
        ),
        "Skill Match (%)": match_data["skill_score"],
        "Experience Match (%)": (
            match_data["experience_score"]
        ),
        "Education Match (%)": (
            match_data["education_score"]
        ),
        "TF-IDF Similarity (%)": (
            match_data["similarity_score"]
        ),
        "Resume Match Score (%)": (
            match_data["match_score"]
        )
    }

    results.append(result)


# Convert to DataFrame
results_df = pd.DataFrame(results)

# Rank candidates
results_df = results_df.sort_values(
    by="Resume Match Score (%)",
    ascending=False
).reset_index(drop=True)

results_df.index += 1

results_df.insert(
    0,
    "Rank",
    results_df.index
)

print("\n🏆 CANDIDATE RANKINGS")

results_df


Total candidates found: 10

🏆 CANDIDATE RANKINGS


,Rank,Name,Experience (Years),Education,Skills,Matched Skills,Missing Skills,Skill Match (%),Experience Match (%),Education Match (%),TF-IDF Similarity (%),Resume Match Score (%)
1,1,Shreya Verma,4.0,Master's,"aws, data science, deep learning, docker, git,...","aws, data science, deep learning, docker, mach...",artificial intelligence,90.91,100.0,100.0,59.87,91.44
2,2,Priya Kapoor,5.0,Master's,"aws, deep learning, docker, fastapi, git, kube...","aws, deep learning, docker, machine learning, ...","artificial intelligence, data science",81.82,100.0,100.0,48.91,85.80
3,3,Aditi Sharma,3.0,Bachelor's,"aws, deep learning, docker, git, machine learn...","aws, deep learning, docker, machine learning, ...","artificial intelligence, data science",81.82,100.0,50.0,65.88,80.00
4,4,Raghav Sharma,3.0,Bachelor's,"aws, deep learning, docker, git, machine learn...","aws, deep learning, docker, machine learning, ...","artificial intelligence, data science",81.82,100.0,50.0,49.21,78.33
5,5,Neha Singh,1.0,Master's,"artificial intelligence, computer vision, data...","artificial intelligence, data science, deep le...","aws, docker, scikit-learn, sql",63.64,50.0,100.0,54.15,64.74
6,6,Rahul Verma,2.0,Bachelor's,"excel, machine learning, numpy, pandas, power ...","machine learning, pandas, python, scikit-learn...","artificial intelligence, aws, data science, de...",45.45,100.0,50.0,44.01,59.63
7,7,Rahul Verma,2.0,Bachelor's,"excel, machine learning, numpy, pandas, power ...","machine learning, pandas, python, scikit-learn...","artificial intelligence, aws, data science, de...",45.45,100.0,50.0,24.04,57.63
8,8,Neha Singh,1.0,Master's,"computer vision, deep learning, git, machine l...","deep learning, machine learning, pandas, pytho...","artificial intelligence, aws, data science, do...",45.45,50.0,100.0,35.73,53.80
9,9,Arjun Mehta,4.0,Bachelor's,"aws, docker, git, java, mysql, python, sql","aws, docker, python, sql","artificial intelligence, data science, deep le...",36.36,100.0,50.0,14.06,52.09
10,10,Karan Malhotra,3.0,Bachelor's,"css, git, github, html, javascript, node.js, r...",,"artificial intelligence, aws, data science, de...",0.00,100.0,50.0,11.26,33.63


In [47]:
print("🔍 MISSING SKILLS ANALYSIS\n")

for _, candidate in results_df.iterrows():

    print(
        f"#{candidate['Rank']} "
        f"{candidate['Name']}"
    )

    print(
        f"Match Score: "
        f"{candidate['Resume Match Score (%)']}%"
    )

    missing = candidate["Missing Skills"]

    if missing:
        print(f"❌ Missing Skills: {missing}")
    else:
        print(
            "✅ No required skills are missing!"
        )

    print("-" * 50)

🔍 MISSING SKILLS ANALYSIS

#1 Shreya Verma
Match Score: 91.44%
❌ Missing Skills: artificial intelligence
--------------------------------------------------
#2 Priya Kapoor
Match Score: 85.8%
❌ Missing Skills: artificial intelligence, data science
--------------------------------------------------
#3 Aditi Sharma
Match Score: 80.0%
❌ Missing Skills: artificial intelligence, data science
--------------------------------------------------
#4 Raghav Sharma
Match Score: 78.33%
❌ Missing Skills: artificial intelligence, data science
--------------------------------------------------
#5 Neha Singh
Match Score: 64.74%
❌ Missing Skills: aws, docker, scikit-learn, sql
--------------------------------------------------
#6 Rahul Verma
Match Score: 59.63%
❌ Missing Skills: artificial intelligence, aws, data science, deep learning, docker, tensorflow
--------------------------------------------------
#7 Rahul Verma
Match Score: 57.63%
❌ Missing Skills: artificial intelligence, aws, data science, dee

In [48]:
SHORTLIST_THRESHOLD = 70

shortlisted_df = results_df[
    results_df["Resume Match Score (%)"]
    >= SHORTLIST_THRESHOLD
].copy()

print(
    f"Shortlisting threshold: "
    f"{SHORTLIST_THRESHOLD}%"
)

print(
    f"Number of shortlisted candidates: "
    f"{len(shortlisted_df)}"
)

shortlisted_df

Shortlisting threshold: 70%
Number of shortlisted candidates: 4


,Rank,Name,Experience (Years),Education,Skills,Matched Skills,Missing Skills,Skill Match (%),Experience Match (%),Education Match (%),TF-IDF Similarity (%),Resume Match Score (%)
1,1,Shreya Verma,4.0,Master's,"aws, data science, deep learning, docker, git,...","aws, data science, deep learning, docker, mach...",artificial intelligence,90.91,100.0,100.0,59.87,91.44
2,2,Priya Kapoor,5.0,Master's,"aws, deep learning, docker, fastapi, git, kube...","aws, deep learning, docker, machine learning, ...","artificial intelligence, data science",81.82,100.0,100.0,48.91,85.80
3,3,Aditi Sharma,3.0,Bachelor's,"aws, deep learning, docker, git, machine learn...","aws, deep learning, docker, machine learning, ...","artificial intelligence, data science",81.82,100.0,50.0,65.88,80.00
4,4,Raghav Sharma,3.0,Bachelor's,"aws, deep learning, docker, git, machine learn...","aws, deep learning, docker, machine learning, ...","artificial intelligence, data science",81.82,100.0,50.0,49.21,78.33


In [49]:
output_file = "shortlisted_candidates.csv"

shortlisted_df.to_csv(
    output_file,
    index=False
)

print(
    f"✅ Shortlisted candidates exported to: "
    f"{output_file}"
)

files.download(output_file)

✅ Shortlisted candidates exported to: shortlisted_candidates.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>